# KVAE Results
## Evaluation and visualization of results for all models.

In [ ]:
import sys
sys.path.append("..")

import torch
import numpy as np
import matplotlib.pyplot as plt

from config.vae_config import VAEConfig
from config.simulation_config import SimulationConfig
from config.train_config import TrainConfig
from models.kvae import KVAE
from models.gru_vae import GRUVAE
from models.cv_vae import CVVAE
from dataset.dataset import BallDataset
from evaluation.evaluate import evaluate, load_model, compute_metrics, compute_mse_per_step, predict_multistep
from utils.visualize import (
    plot_trajectories,
    plot_reconstruction_grid,
    plot_alpha,
    plot_prediction_mse,
    plot_latent_space,
)

device  = torch.device("cuda" if torch.cuda.is_available() else "cpu")
cfg     = VAEConfig()
sim_cfg = SimulationConfig()
tcfg    = TrainConfig()

## Loading models

In [ ]:
CHECKPOINTS = {
    "KVAE":    "checkpoints/kvae/best.pt",
    "GRU-VAE": "checkpoints/gru_vae/best.pt",
    "CV-VAE":  "checkpoints/cv_vae/best.pt",
}

models = {
    "KVAE":    load_model(CHECKPOINTS["KVAE"],    "kvae",    cfg, sim_cfg, device),
    "GRU-VAE": load_model(CHECKPOINTS["GRU-VAE"], "gru_vae", cfg, sim_cfg, device),
    "CV-VAE":  load_model(CHECKPOINTS["CV-VAE"],  "cv_vae",  cfg, sim_cfg, device),
}

for name, model in models.items():
    print(f"{name}: {model}")

## Dataset

In [ ]:
from torch.utils.data import DataLoader

test_dataset = BallDataset(sim_cfg, tcfg, split="test")
test_loader  = DataLoader(test_dataset, batch_size=32, shuffle=False, num_workers=4)

print(f"Test samples: {len(test_dataset)}")

## Metrics

### Comparison of all models on the test set.

In [ ]:
import pandas as pd
import os

MAX_PRED_STEPS = 10
N_FREE_STEPS   = 10

all_metrics     = {}
all_mse_per_step = {}

for name, model in models.items():
    print(f"Evaluating {name}...")
    metrics = compute_metrics(model, test_loader, device)
    mse_per_step = compute_mse_per_step(model, test_loader, MAX_PRED_STEPS, device)

    all_metrics[name]      = metrics
    all_mse_per_step[name] = mse_per_step

rows = []
for name, m in all_metrics.items():
    row = {"Model": name}
    row.update({k: f"{v:.6f}" for k, v in m.items()})
    for n in [1, 5, 10]:
        row[f"Pred MSE ({n})"] = f"{all_mse_per_step[name][n-1]:.6f}"
    rows.append(row)

df = pd.DataFrame(rows).set_index("Model")
display(df)

os.makedirs("results", exist_ok=True)
df.to_csv("results/metrics_table.csv")
print("Sacuvano: results/metrics_table.csv")

## MSE vs Prediction Horizon

In [ ]:
plot_prediction_mse(
    mse_per_step_kvae = all_mse_per_step["KVAE"],
    mse_per_step_vae = all_mse_per_step["GRU-VAE"],
    save_path = "results/plots/mse_per_step.png"
)

fig, ax = plt.subplots(figsize=(8, 5))
colors = {"KVAE": "seagreen", "GRU-VAE": "steelblue", "CV-VAE": "tomato"}
styles = {"KVAE": "-", "GRU-VAE": "--", "CV-VAE": ":"}
steps  = np.arange(1, MAX_PRED_STEPS + 1)

for name, mse in all_mse_per_step.items():
    ax.plot(steps, mse, label=name,
            color=colors[name], linestyle=styles[name],
            linewidth=2.0, marker="o", markersize=5)

ax.set_xlabel("Prediction horizon (steps)", fontsize=12)
ax.set_ylabel("MSE",                        fontsize=12)
ax.set_title("Prediction MSE vs Horizon",   fontsize=14)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("results/plots/mse_per_step_all.png", dpi=150, bbox_inches="tight")
plt.show()

## Visualizations by Sample
### The same sample from the test set is displayed for each model.

In [ ]:
SAMPLE_IDX   = 0
N_FREE_STEPS = 10

ball_seq, obstacle_img = test_dataset[SAMPLE_IDX]
ball_seq     = ball_seq.unsqueeze(0).to(device)
obstacle_img = obstacle_img.unsqueeze(0).to(device)

outputs = {}
for name, model in models.items():
    outputs[name] = predict_multistep(model, ball_seq, obstacle_img, N_FREE_STEPS, device)
    print(f"{name} — done")

### Trajectories in latent space

In [ ]:
fig, axes = plt.subplots(1, len(models), figsize=(7 * len(models), 6))

colors = {"KVAE": "seagreen", "GRU-VAE": "steelblue", "CV-VAE": "tomato"}

for ax, (name, out) in zip(axes, outputs.items()):
    ax.plot(out["a_mu"][:, 0],   out["a_mu"][:, 1],
            color="gray", linewidth=1.0, alpha=0.5, label="Encoder")
    ax.plot(out["a_filt"][:, 0], out["a_filt"][:, 1],
            color=colors[name], linewidth=2.0, label="Filtered")
    ax.plot(out["a_pred_free"][:, 0], out["a_pred_free"][:, 1],
            color="tomato", linewidth=2.0, linestyle="--", label="Free-running")
    ax.scatter(*out["a_filt"][0],  color=colors[name], s=80, zorder=5, marker="o")
    ax.scatter(*out["a_filt"][-1], color=colors[name], s=80, zorder=5, marker="X")
    ax.set_title(name, fontsize=13)
    ax.set_xlabel("$a_1$")
    ax.set_ylabel("$a_2$")
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)

plt.suptitle("Latent Space Trajectories", fontsize=15)
plt.tight_layout()
plt.savefig("results/plots/trajectories_all.png", dpi=150, bbox_inches="tight")
plt.show()

### Frames reconstruction

In [ ]:
for name, out in outputs.items():
    print(f"\\n{name}")
    plot_reconstruction_grid(
        ball_seq   = out["ball_seq"],
        x_hat_filt = out["x_hat_filt"],
        x_hat_pred = out["x_hat_pred"],
        save_path  = f"results/plots/reconstruction_{name.lower().replace('-','_')}.png"
    )

### Alpha Weights Over Time (KVAE)

In [ ]:
if outputs["KVAE"]["alpha_seq"] is not None:
    plot_alpha(
        alpha_seq = outputs["KVAE"]["alpha_seq"],
        save_path = "results/plots/alpha_kvae.png"
    )

### Latent space: Encoder vs Kalman

In [ ]:
for name, out in outputs.items():
    print(f"\\n{name}")
    plot_latent_space(
        a_mu   = out["a_mu"],
        a_filt = out["a_filt"],
        save_path = f"results/plots/latent_{name.lower().replace('-','_')}.png"
    )

## Multi-Sample Visualizations

In [ ]:
N_SAMPLES = 5

for i in range(N_SAMPLES):
    ball_seq_i, obstacle_img_i = test_dataset[i]
    ball_seq_i     = ball_seq_i.unsqueeze(0).to(device)
    obstacle_img_i = obstacle_img_i.unsqueeze(0).to(device)

    out = predict_multistep(models["KVAE"], ball_seq_i, obstacle_img_i, N_FREE_STEPS, device)

    plot_trajectories(
        a_mu        = out["a_mu"],
        a_filt      = out["a_filt"],
        a_pred_free = out["a_pred_free"],
        save_path   = f"results/plots/sample_{i}/trajectories.png"
    )

    plot_reconstruction_grid(
        ball_seq   = out["ball_seq"],
        x_hat_filt = out["x_hat_filt"],
        x_hat_pred = out["x_hat_pred"],
        save_path  = f"results/plots/sample_{i}/reconstruction.png"
    )